# Endo-Depth-and-Motion — Colab 完整运行手册

## ⚠️ 运行前检查
1. 运行时 → 更改运行时类型 → **T4 GPU**
2. 需要提前准备好 Google Drive 文件夹结构：
```
我的云端硬盘/Endo-Depth/
  data/         ← 放解压后的 test1, test17... 文件夹
  results/      ← 放 Hamlyn_test_odometries.zip 解压后的 .pkl 文件
  models/       ← 放预训练模型（如需跑深度估计）
```

## 已知问题总结
| 问题 | 原因 | 解决方案 |
|------|------|----------|
| pytorch3d 无预编译 wheel | PyTorch 2.x + Python 3.12 太新 | 从源码编译，需 15~25 分钟 |
| pandas==1.0.3 安装失败 | Python 3.12 不兼容 | 不指定版本，装最新版 |
| kornia==0.5.11 安装失败 | 版本太旧 | 装最新版 |
| tracking 脚本无法在 Colab 跑 | 需要 GUI 窗口和键盘交互 | 使用预计算的 odometry 直接跑 fusion |
| Plotly 在 Colab 不显示 | Colab 渲染限制 | 保存为 HTML 下载到本地浏览器打开 |
| zip 文件干扰 list_files | 代码把 zip 也当目录扫描 | 解压后删掉 zip 文件 |
| test21 缺少 intrinsics.txt | 数据不完整 | 跳过或忽略 |

## Step 0：挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 1：验证 GPU 环境

In [ ]:
import torch, sys
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU available:', torch.cuda.is_available())
# 期望输出：PyTorch 2.x, CUDA 12.x, GPU available: True

## Step 2：安装依赖（首次运行约 20~30 分钟）

In [ ]:
# 2.1 降级 PyTorch 到有预编译 pytorch3d wheel 的版本
# 注意：会有 torchaudio 冲突警告，忽略即可
!pip install torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121 -q

In [ ]:
# 2.2 安装编译 pytorch3d 所需依赖
!pip install ninja fvcore iopath -q

In [ ]:
# 2.3 从源码编译安装 pytorch3d（最慢的一步，15~25 分钟，有输出说明在跑）
!MAX_JOBS=4 pip install "git+https://github.com/facebookresearch/pytorch3d.git" \
  --no-build-isolation 2>&1 | grep -E '(building|error|Running|Successfully|FAILED)'

In [ ]:
# 2.4 安装其余依赖
# 注意：pandas/kornia 不写版本号，旧版本与 Python 3.12 不兼容
!pip install \
  tqdm==4.43.0 \
  plotly==4.5.4 \
  pandas \
  "numpy>=1.18.2" \
  tensorboard==2.2.1 \
  kornia \
  Pillow==8.4.0 \
  logzero==1.5.0 \
  pytest==5.4.2 \
  imageio==2.8.0 \
  matplotlib==3.2.1 \
  codetiming==1.2.0 \
  open3d \
  PyGeometry -q

In [ ]:
# 2.5 验证所有关键包
import torch, pytorch3d, open3d, kornia, cv2
print('torch:', torch.__version__)
print('pytorch3d:', pytorch3d.__version__)
print('open3d:', open3d.__version__)
print('kornia:', kornia.__version__)
print('CUDA available:', torch.cuda.is_available())

## Step 3：Clone 项目代码（每次重启 Colab 需重新执行）

In [ ]:
%cd /content
!git clone https://github.com/dundun12334-coder/Endo-Depth-and-Motion.git
%cd /content/Endo-Depth-and-Motion
!ls

## Step 4：准备数据

**需要提前上传到 Google Drive：**
- Hamlyn tracking 测试数据（zip）→ 解压到 `Endo-Depth/data/`
- 预计算 odometry（Hamlyn_test_odometries.zip）→ 解压到 `Endo-Depth/results/`

数据下载链接（需要 OneDrive）：
- 测试数据: https://unizares-my.sharepoint.com/:f:/g/personal/recasens_unizar_es/Epwqt3JCs4BJnEiV9esUH0gBeJYbTmmNCouEpncW4MjC8A
- Odometry: https://unizares-my.sharepoint.com/:f:/g/personal/recasens_unizar_es/EmskdlBSuTlHk2B13S37QpoBx1sdXXpzAdDOUMxSGMW_kA

In [ ]:
import zipfile, os

# 解压测试数据
data_dir = '/content/drive/MyDrive/Endo-Depth/data'
os.makedirs(data_dir, exist_ok=True)

for fname in os.listdir(data_dir):
    if fname.endswith('.zip'):
        fpath = os.path.join(data_dir, fname)
        try:
            with zipfile.ZipFile(fpath, 'r') as z:
                z.extractall(data_dir)
            print(f'✓ {fname} 解压成功')
        except zipfile.BadZipFile:
            size = os.path.getsize(fpath) / 1024 / 1024
            print(f'✗ {fname} 无效（{size:.1f}MB，可能未上传完）')

In [ ]:
# 删掉 zip 文件（避免干扰 list_files）
for fname in os.listdir(data_dir):
    if fname.endswith('.zip'):
        os.remove(os.path.join(data_dir, fname))
        print(f'删除 {fname}')

# 验证目录结构
!find /content/drive/MyDrive/Endo-Depth/data -maxdepth 3 -type d

In [ ]:
# 解压 odometry
results_dir = '/content/drive/MyDrive/Endo-Depth/results'
os.makedirs(results_dir, exist_ok=True)

odo_zip = '/content/drive/MyDrive/Endo-Depth/Hamlyn_test_odometries.zip'
if os.path.exists(odo_zip):
    with zipfile.ZipFile(odo_zip, 'r') as z:
        z.extractall(results_dir)
    print('✓ Odometry 解压完成')
    !find /content/drive/MyDrive/Endo-Depth/results -name '*.pkl'
else:
    print('⚠️ 未找到 odometry zip，请先上传到 Drive')

## Step 5：Volumetric Fusion（生成 3D mesh）

In [ ]:
# 注释掉 GUI 那行（Colab 无法显示 Open3D 窗口）
!sed -i 's/    o3d.visualization.draw_geometries/#   o3d.visualization.draw_geometries/' \
  /content/Endo-Depth-and-Motion/apps/volumetric_fusion/__main__.py
print('✓ 已注释 GUI 代码')

In [ ]:
# 跑单个场景（test1）
# 修改 pkl 路径和数据路径即可跑其他场景
PKL = '/content/drive/MyDrive/Endo-Depth/results/Hamlyn_test_odometries/1.pkl'
DATA = '/content/drive/MyDrive/Endo-Depth/data/test1'

!python /content/Endo-Depth-and-Motion/apps/volumetric_fusion/__main__.py \
  -i {PKL} \
  -o {DATA}

In [ ]:
# 批量跑所有场景
import glob

pkl_files = sorted(glob.glob('/content/drive/MyDrive/Endo-Depth/results/Hamlyn_test_odometries/*.pkl'))
print(f'找到 {len(pkl_files)} 个场景')

for pkl in pkl_files:
    scene_id = os.path.splitext(os.path.basename(pkl))[0]
    data_path = f'/content/drive/MyDrive/Endo-Depth/data/test{scene_id}'
    if os.path.exists(data_path):
        print(f'\n--- 处理 test{scene_id} ---')
        !python /content/Endo-Depth-and-Motion/apps/volumetric_fusion/__main__.py \
          -i {pkl} -o {data_path}
    else:
        print(f'跳过 test{scene_id}（数据不存在）')

## Step 6：可视化 3D Mesh

In [ ]:
import open3d as o3d
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'colab'

# 修改这里选择要可视化的场景
PLY_PATH = '/content/drive/MyDrive/Endo-Depth/data/test1/map/1/mesh.ply'

mesh = o3d.io.read_triangle_mesh(PLY_PATH)
print(f'顶点数: {len(mesh.vertices)}')
print(f'三角面数: {len(mesh.triangles)}')

vertices = np.asarray(mesh.vertices)
triangles = np.asarray(mesh.triangles)
colors = np.asarray(mesh.vertex_colors)

fig = go.Figure(data=[go.Mesh3d(
    x=vertices[:, 0],
    y=vertices[:, 1],
    z=vertices[:, 2],
    i=triangles[:, 0],
    j=triangles[:, 1],
    k=triangles[:, 2],
    vertexcolor=['rgb({},{},{})'.format(int(r*255), int(g*255), int(b*255))
                 for r, g, b in colors] if len(colors) > 0 else None,
    opacity=1.0,
    lighting=dict(ambient=0.8)
)])
fig.update_layout(
    scene=dict(aspectmode='data'),
    width=900, height=600,
    title=f'3D Mesh: {PLY_PATH.split("/")[-3]}'
)

# 保存 HTML（下载到本地浏览器打开效果更好）
html_path = PLY_PATH.replace('mesh.ply', 'mesh_viz.html')
fig.write_html(html_path)
print(f'✓ HTML 已保存: {html_path}')

fig.show()

## Step 7（可选）：深度估计

如果你有自己的内窥镜图片/视频（没有深度图），需要先下载预训练模型：
- 模型下载: https://unizares-my.sharepoint.com/:f:/g/personal/recasens_unizar_es/EmBjII1JZ9RJntKgoai8a_8BPvqyY02w1S43vQoNTiQh8Q
- 下载后上传到 `Endo-Depth/models/`

In [ ]:
# 深度估计（需要先下载模型）
IMAGE_DIR = '/content/drive/MyDrive/Endo-Depth/your_images'   # ← 修改为你的图片路径
MODEL_DIR = '/content/drive/MyDrive/Endo-Depth/models'        # ← 修改为模型路径
OUTPUT_DIR = '/content/drive/MyDrive/Endo-Depth/depth_output'

!python /content/Endo-Depth-and-Motion/apps/depth_estimate/__main__.py \
  --image_path {IMAGE_DIR} \
  --model_path {MODEL_DIR} \
  --output_path {OUTPUT_DIR} \
  --output_type color

## 注意事项

1. **每次重启 Colab** 需要重新执行 Step 2（安装依赖）和 Step 3（clone 代码）
2. **Plotly 在 Colab 有时不显示**，直接下载 HTML 文件到本地浏览器打开
3. **tracking 脚本无法在 Colab 跑**（需要 GUI），使用预计算的 odometry 代替
4. **test15 数据损坏**（zip 只有 2.4MB），跳过即可
5. **test21 缺少 intrinsics.txt**，跑 volumetric fusion 时可能出错